# Stage 1 — Base 预测器与 H1 检验

> 对应 §8.2、§10.1、§13.3。

**H1 是决定整个项目有没有立足点的实验，一天能跑完。**
只需 Base + 朴素 kNN：不需要门控、不需要重排器、不需要图匹配、不需要交叉拟合。

止损：若 ≥2/3 的 Tier-1 靶点上低支持子集 Δℓ 的 95% CI 下界 ≤ 0 ⇒ 立即执行 Plan-B。


In [ ]:
# 让 notebook 能 import sparc（无需 pip install -e .）
import sys, json
from pathlib import Path
CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))

from sparc.common import load_experiment_config
cfg = load_experiment_config(stage='notebook', run_name='interactive')
print('冻结配置指纹：'); print(json.dumps(cfg.freeze_manifest(), indent=1))


## 1. 参数预算核对（无需 torch 也能验）


In [ ]:
from sparc.models.param_budget import full_budget, supervision_ratio

budget = full_budget(cfg.hparams.dims)
frozen = cfg.hparams.param_budget
for key in ['theta_b','graphmatcher','reranker','evidence','residual','gate','uncertainty_head','theta_r','total_trainable']:
    mark = 'OK' if budget[key] == frozen[key] else 'MISMATCH'
    print(f'{key:18s}{budget[key]:>9d}{frozen[key]:>9d}  {mark}')
print()
print(supervision_ratio(budget['theta_r'], n_train_queries=3510))


## 2. 训练 Θ_B + H1

```bash
python scripts/run_s1_base.py --run-name s1_v1 --model-a molformer_xl
```


In [ ]:
h1_path = cfg.paths.stage_outputs('s1_base') / 's1_v1_h1.json'
if h1_path.is_file():
    h1 = json.loads(h1_path.read_text(encoding='utf-8'))
    print('H1 通过 =', h1['passed'])
    for name, c in h1['criteria'].items():
        print(f"  {name:24s} 值={c.get('value')}  阈值={c.get('threshold')}  {'✓' if c['passed'] else '✗'}")
    print(h1['note'])
else:
    print('尚未跑 Stage 1。')


## 3. 训练曲线


In [ ]:
from sparc.common.logging_utils import MetricMonitor

records = MetricMonitor(cfg.paths.stage_logs('s1_base'), 's1_v1').read_all()
if records:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot([r['epoch'] for r in records], [r['train_loss'] for r in records], label='train')
    ax[0].plot([r['epoch'] for r in records], [r['val_metric'] for r in records], label='val')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('Tobit NLL'); ax[0].legend(); ax[0].grid(alpha=.2)
    ax[1].plot([r['epoch'] for r in records], [r.get('val_rmse') for r in records], color='#B91C1C')
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val RMSE'); ax[1].grid(alpha=.2)
    plt.tight_layout()
else:
    print('无训练记录。')
